In [1]:
import pandas as pd
import numpy as np
import umap
import glob
import os
import sys
import joblib
from pathlib import Path
import pickle
from scipy.io import readsav

sys.path.insert(0, "cellular_cycle") # add path to the model

# should examine these functions to determine which are best to use (overlapping functionality)
from model_fxns import minmax_scale, stack_sliding_window, calc_minmax, load_sim_output, load_norm_data


In [2]:
SAVE_DIR = Path(Path.home(),'ECM_mod_output/UMAP/ODE_embeddings')
SAVE_DIR.mkdir(parents=True, exist_ok=True)

RED_DIR = Path(Path.home(),'ECM_mod_output/UMAP') # directory for UMAP reducer object

OVERWRITE = True
SET_JPARAM = True
PSO_NUM = 2

sys_type = sys.platform

In [3]:
FILE_LOC = Path(Path.home(),'ECM_mod_output/ODE_output')
if SET_JPARAM:
    filenames = [Path(FILE_LOC,f) for f in os.listdir(FILE_LOC) if  f'psoparam_{PSO_NUM}' in f]
else:
    filenames = [Path(FILE_LOC,f) for f in os.listdir(FILE_LOC) if  'range' in f]    
    
all_odes = sorted(filenames)

In [4]:
all_odes[:5]

[WindowsPath('C:/Users/olesk/ECM_mod_output/ODE_output/DMSO_psoparam_2_000.csv'),
 WindowsPath('C:/Users/olesk/ECM_mod_output/ODE_output/DMSO_psoparam_2_001.csv'),
 WindowsPath('C:/Users/olesk/ECM_mod_output/ODE_output/DMSO_psoparam_2_002.csv'),
 WindowsPath('C:/Users/olesk/ECM_mod_output/ODE_output/DMSO_psoparam_2_003.csv'),
 WindowsPath('C:/Users/olesk/ECM_mod_output/ODE_output/DMSO_psoparam_2_004.csv')]

In [5]:
assert len(all_odes) == 2000

In [6]:
# Important params to set the window size and step time distance
init_dt = 0.1                # initial dt from the raw model output
dt = 0.1                      # dt for umap window
window = 4                    # window size in hours
time_step = 0.5               # time step between windows

#umap params
neighbs = 50
min_d = 0.05



In [7]:
if sys_type == 'darwin':
    sys_name = 'mac'
    r_path_date = '20240730'
    r_dbpath = 'https://www.dropbox.com/scl/fi/pvx434xg8e391bil5b8wd/reducer_mac_0.1_0.1_4_0.5.sav?rlkey=nuwm98ld3guyukz14a9m6ewu7&dl=1'
if sys_type == 'win32': 
    sys_name = 'win'
    r_path_date = '20240820'
    r_dbpath = 'https://www.dropbox.com/s/a0jo15tvh70wrjv/reducer_win_0.1_0.1_4_0.5.sav?dl=1'
if sys_type == 'linux': 
    sys_name = 'linux'
    r_path_date = '20240606'
    r_dbpath = "https://www.dropbox.com/scl/fi/79lp5174wny1ngiy4w1h7/reducer_20240606_0.1_0.1_4_0.5.sav?rlkey=3p1xkbzkdmwy9ivmn5r64xtlh&dl=1"

if SET_JPARAM:
    r_name = f'reducerPSO_{PSO_NUM}_{sys_name}_{init_dt}_{dt}_{window}_{time_step}.sav'
else:
    r_name = f'reducer_{sys_name}_{init_dt}_{dt}_{window}_{time_step}.sav'

In [8]:
print(f'Reducer filename: {r_name}')

Reducer filename: reducerPSO_2_win_0.1_0.1_4_0.5.sav


In [9]:
r_path = Path(SAVE_DIR,'UMAP',r_name)

%env R_PATH = {r_path}
%env R_DBPATH = {r_dbpath}

env: R_PATH=C:\Users\olesk\ECM_mod_output\UMAP\ODE_embeddings\UMAP\reducerPSO_2_win_0.1_0.1_4_0.5.sav
env: R_DBPATH=https://www.dropbox.com/s/a0jo15tvh70wrjv/reducer_win_0.1_0.1_4_0.5.sav?dl=1


In [10]:
r_path = Path(RED_DIR,r_name)

if not r_path.exists():
    %env R_DBPATH = {r_dbpath}
    print(f'Reducer file not found, downloading from Dropbox: {r_dbpath}')
    !wget -O $R_PATH $R_DBPATH
print(f'Loading reducer from {r_path}')
reducer = joblib.load(r_path)

Loading reducer from C:\Users\olesk\ECM_mod_output\UMAP\reducerPSO_2_win_0.1_0.1_4_0.5.sav


In [11]:
exp_stiff_vals = ['000','049','099']
SAVE_DIR = Path(Path.home(),'ECM_mod_output')

In [12]:
if SET_JPARAM:
    stoch_filenames = [Path(SAVE_DIR,f) for f in os.listdir(SAVE_DIR) if 'stoch_sim' in f and f'pso_{PSO_NUM}' in f and not 'orig' in f]
    stoch_filenames.sort()
else:
    stoch_filenames = [Path(SAVE_DIR,f) for f in os.listdir(SAVE_DIR) if 'stoch_sim' in f and '_stiff_' in f and 'orig' in f]
    stoch_filenames.sort()


stoch_filenames.sort()
stoch_filenames

[WindowsPath('C:/Users/olesk/ECM_mod_output/stoch_sim_DMSO_stiff_000_pso_2.csv'),
 WindowsPath('C:/Users/olesk/ECM_mod_output/stoch_sim_DMSO_stiff_029_pso_2.csv'),
 WindowsPath('C:/Users/olesk/ECM_mod_output/stoch_sim_DMSO_stiff_049_pso_2.csv'),
 WindowsPath('C:/Users/olesk/ECM_mod_output/stoch_sim_DMSO_stiff_074_pso_2.csv'),
 WindowsPath('C:/Users/olesk/ECM_mod_output/stoch_sim_DMSO_stiff_099_pso_2.csv'),
 WindowsPath('C:/Users/olesk/ECM_mod_output/stoch_sim_ERL_stiff_000_pso_2.csv'),
 WindowsPath('C:/Users/olesk/ECM_mod_output/stoch_sim_ERL_stiff_029_pso_2.csv'),
 WindowsPath('C:/Users/olesk/ECM_mod_output/stoch_sim_ERL_stiff_049_pso_2.csv'),
 WindowsPath('C:/Users/olesk/ECM_mod_output/stoch_sim_ERL_stiff_074_pso_2.csv'),
 WindowsPath('C:/Users/olesk/ECM_mod_output/stoch_sim_ERL_stiff_099_pso_2.csv')]

In [13]:
stoch_filenames_umap = [f for f in stoch_filenames if any([x in str(f) for x in exp_stiff_vals])]
stoch_filenames_umap

[WindowsPath('C:/Users/olesk/ECM_mod_output/stoch_sim_DMSO_stiff_000_pso_2.csv'),
 WindowsPath('C:/Users/olesk/ECM_mod_output/stoch_sim_DMSO_stiff_049_pso_2.csv'),
 WindowsPath('C:/Users/olesk/ECM_mod_output/stoch_sim_DMSO_stiff_099_pso_2.csv'),
 WindowsPath('C:/Users/olesk/ECM_mod_output/stoch_sim_ERL_stiff_000_pso_2.csv'),
 WindowsPath('C:/Users/olesk/ECM_mod_output/stoch_sim_ERL_stiff_049_pso_2.csv'),
 WindowsPath('C:/Users/olesk/ECM_mod_output/stoch_sim_ERL_stiff_099_pso_2.csv')]

In [14]:
ode_filenames = [Path(str(f).replace('stoch_sim','ODE_sim')) for f in stoch_filenames]
ode_filenames

[WindowsPath('C:/Users/olesk/ECM_mod_output/ODE_sim_DMSO_stiff_000_pso_2.csv'),
 WindowsPath('C:/Users/olesk/ECM_mod_output/ODE_sim_DMSO_stiff_029_pso_2.csv'),
 WindowsPath('C:/Users/olesk/ECM_mod_output/ODE_sim_DMSO_stiff_049_pso_2.csv'),
 WindowsPath('C:/Users/olesk/ECM_mod_output/ODE_sim_DMSO_stiff_074_pso_2.csv'),
 WindowsPath('C:/Users/olesk/ECM_mod_output/ODE_sim_DMSO_stiff_099_pso_2.csv'),
 WindowsPath('C:/Users/olesk/ECM_mod_output/ODE_sim_ERL_stiff_000_pso_2.csv'),
 WindowsPath('C:/Users/olesk/ECM_mod_output/ODE_sim_ERL_stiff_029_pso_2.csv'),
 WindowsPath('C:/Users/olesk/ECM_mod_output/ODE_sim_ERL_stiff_049_pso_2.csv'),
 WindowsPath('C:/Users/olesk/ECM_mod_output/ODE_sim_ERL_stiff_074_pso_2.csv'),
 WindowsPath('C:/Users/olesk/ECM_mod_output/ODE_sim_ERL_stiff_099_pso_2.csv')]

In [15]:
ode_filenames_umap = [Path(str(f).replace('stoch_sim','ODE_sim')) for f in stoch_filenames_umap]
ode_filenames_umap

[WindowsPath('C:/Users/olesk/ECM_mod_output/ODE_sim_DMSO_stiff_000_pso_2.csv'),
 WindowsPath('C:/Users/olesk/ECM_mod_output/ODE_sim_DMSO_stiff_049_pso_2.csv'),
 WindowsPath('C:/Users/olesk/ECM_mod_output/ODE_sim_DMSO_stiff_099_pso_2.csv'),
 WindowsPath('C:/Users/olesk/ECM_mod_output/ODE_sim_ERL_stiff_000_pso_2.csv'),
 WindowsPath('C:/Users/olesk/ECM_mod_output/ODE_sim_ERL_stiff_049_pso_2.csv'),
 WindowsPath('C:/Users/olesk/ECM_mod_output/ODE_sim_ERL_stiff_099_pso_2.csv')]

In [16]:
# max_prots, min_prots = calc_minmax(stoch_filenames, ode_filenames)
max_prots, min_prots = calc_minmax(stoch_filenames_umap + ode_filenames_umap)

In [17]:
STOCH_LOC = Path(Path.home(),'ECM_mod_output/')
if SET_JPARAM:
    stoch_filenames = [Path(SAVE_DIR,f) for f in os.listdir(SAVE_DIR) if 'stoch_sim' in f and f'pso_{PSO_NUM}' in f and not 'orig' in f]
else:
    filenames = [Path(STOCH_LOC,f) for f in os.listdir(STOCH_LOC) if 'stoch_sim' in f and 'cycdi_0_0' in f and not 'psoparam' in f]
filenames = sorted(filenames)

filenames_conds = [filenames[2], filenames[3], filenames[0],
                filenames[1], filenames[4], filenames[5]]

filenames_conds

[WindowsPath('C:/Users/olesk/ECM_mod_output/ODE_output/DMSO_psoparam_2_002.csv'),
 WindowsPath('C:/Users/olesk/ECM_mod_output/ODE_output/DMSO_psoparam_2_003.csv'),
 WindowsPath('C:/Users/olesk/ECM_mod_output/ODE_output/DMSO_psoparam_2_000.csv'),
 WindowsPath('C:/Users/olesk/ECM_mod_output/ODE_output/DMSO_psoparam_2_001.csv'),
 WindowsPath('C:/Users/olesk/ECM_mod_output/ODE_output/DMSO_psoparam_2_004.csv'),
 WindowsPath('C:/Users/olesk/ECM_mod_output/ODE_output/DMSO_psoparam_2_005.csv')]

In [18]:
if SET_JPARAM:
    filenames_ode = [Path(STOCH_LOC,f) for f in os.listdir(STOCH_LOC) if 'ODE_sim' in f and 'psoparam' in f]
else:
    filenames_ode = [Path(STOCH_LOC,f) for f in os.listdir(STOCH_LOC) if 'ODE_sim' in f and 'cycdi_0_0' in f and not 'psoparam' in f]
    
filenames_ode = sorted(filenames_ode)

filenames_conds_ode = [filenames_ode[2], filenames_ode[3], filenames_ode[0],
                filenames_ode[1], filenames_ode[4], filenames_ode[5]]

filenames_conds_ode


[WindowsPath('C:/Users/olesk/ECM_mod_output/ODE_sim_soft_DMSO_psoparam_cycdi_0_0.csv'),
 WindowsPath('C:/Users/olesk/ECM_mod_output/ODE_sim_soft_erl_psoparam_cycdi_0_0.csv'),
 WindowsPath('C:/Users/olesk/ECM_mod_output/ODE_sim_int_DMSO_psoparam_cycdi_0_0.csv'),
 WindowsPath('C:/Users/olesk/ECM_mod_output/ODE_sim_int_erl_psoparam_cycdi_0_0.csv'),
 WindowsPath('C:/Users/olesk/ECM_mod_output/ODE_sim_stiff_DMSO_psoparam_cycdi_0_0.csv'),
 WindowsPath('C:/Users/olesk/ECM_mod_output/ODE_sim_stiff_erl_psoparam_cycdi_0_0.csv')]

In [19]:
max_prots, min_prots = calc_minmax(filenames + filenames_ode)

In [20]:
SAVE_DIR = Path(Path.home(),'ECM_mod_output/UMAP/ODE_embeddings')
win_len = int(window/dt)
all_prot_tmp = np.empty((0,win_len*18))     #careful, this could end up being massive 
all_prot_tmp_ode = np.empty((0,win_len*18))
all_time_tmp = np.empty((0,win_len))
all_time_at_tmp = np.empty((0,1))
#range_ind = np.arange(int(win_len/2),int(len(time_tmp2)-(int(win_len/2))),int(time_step/dt))

for f in all_odes:
    prot_tmp = pd.read_csv(f, sep=",", header=None, usecols = range(0,19))
    prot_tmp.columns = ["skp2", "p21", "pp21", "cdh1_act", "cdh1_inact", "myc", "cycD", "cycD-p21", 
    "Rb", "pRb", "E2F", "E2F-Rb", "cycE", "cycE-p21", "cycA", "cycA-p21",
    "cycB", "cycB-p21", "cdc20"]
    prot_tmp = prot_tmp.drop(columns=['cdh1_inact'])
    
    range_ind = np.arange(int(win_len/2),int(len(prot_tmp)-(int(win_len/2))),int(time_step/dt))
    #prot_tmp = prot_tmp/200
    for ii in range(18):
        prot_tmp.iloc[:,ii] = (prot_tmp.iloc[:,ii] - min_prots[ii])/(max_prots[ii] - min_prots[ii])

    prot_re_tmp = np.empty((len(range_ind),win_len*18))
    time_re_tmp = np.empty((len(range_ind),win_len))
    time_at_tmp = np.empty((len(range_ind),1))

    ii=0
    for n in range(int(win_len/2),int(len(prot_tmp)-(int(win_len/2))),int(time_step/dt)):
        tmp = prot_tmp.iloc[n-int(win_len/2):n+int(win_len/2)]
        #tmp_t = time_tmp2['time'].iloc[n-int(win_len/2):n+int(win_len/2)]
        tmp=tmp.to_numpy()
        tmp=np.reshape(np.transpose(tmp),win_len*18)
        #tmp_at = time_tmp2['time'].iloc[n]

        prot_re_tmp[ii,:] = np.array([tmp])
        #time_re_tmp[ii,:] = np.array([tmp_t])
        #time_at_tmp[ii,:] = np.array([[tmp_at]])
        ii+=1

    embedding_tmp = reducer.transform(prot_re_tmp)
    absolute_path = f.absolute().as_posix()    
    split_s = absolute_path.split('.')
    split_ss = split_s[0].split('/')

    name = 'embedding_'+split_ss[-1]+'.csv'
    
    np.savetxt(Path(SAVE_DIR, name), embedding_tmp, delimiter=',', fmt='%f')



